<a href="https://colab.research.google.com/github/hardtime-1/hardtime-1/blob/main/imp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1: Check GPU Availability
import torch
print("GPU available:", torch.cuda.is_available())


GPU available: False


In [ ]:
# STEP 2: Install Dependencies
!pip install torch torchvision torchaudio opencv-python numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [13]:
# STEP 3: Install Dependencies
!pip install torch torchvision torchaudio opencv-python numpy matplotlib

# STEP: Download ACOD-12K from Hugging Face
!wget --content-disposition -O ACOD-12K.zip "https://huggingface.co/datasets/Kki11/ACOD-12K/resolve/main/ACOD-12K.zip"

# STEP : Extract the Dataset
!unzip ACOD-12K.zip -d /content/dataset/

# STEP : Verify the Files
!ls /content/dataset/ACOD-12K/

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00539.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00540.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00541.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00542.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00543.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00544.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00545.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00546.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00547.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00548.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00549.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left_00550.png  
  inflating: /content/dataset/ACOD-12K/Train/GT/zucchini_left

In [3]:
# STEP 4: Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import cv2
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader


In [4]:
# STEP 5: Use GPU if Available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# STEP 6: Define RISNet Model (Optimized for Colab)
class RISNet(nn.Module):
    def __init__(self):
        super(RISNet, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [6]:
# STEP 7: Load Model onto GPU
model = RISNet().to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# STEP 8: Define Dataset Class
class ConcealedCropDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        image = cv2.resize(image, (256, 256))
        mask = cv2.resize(mask, (256, 256))

        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0
        mask = torch.tensor(mask).unsqueeze(0).float() / 255.0

        return image, mask

In [8]:
# STEP 9: Define Dataset Paths
dataset_path = "dataset/ACOD-12K/"
train_images = os.path.join(dataset_path, "Train/Imgs/")
train_masks = os.path.join(dataset_path, "Train/GT/")


In [9]:
# STEP 10: Create DataLoader
train_dataset = ConcealedCropDataset(train_images, train_masks)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

In [10]:
# STEP 11: Define Loss Function & Optimizer
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [12]:
# STEP 12: Train Model (Runs on GPU)
# Set the number of epochs
num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    epoch_loss = 0  # Initialize loss for this epoch

    for images, masks in train_loader:
        # Move images and masks to the GPU (if available)
        images, masks = images.to(device), masks.to(device)

        # Ensure target masks have the same size as model output
        masks = torch.nn.functional.interpolate(masks, size=(64, 64), mode="bilinear", align_corners=True)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute the loss
        loss = loss_fn(outputs, masks)

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Accumulate epoch loss
        epoch_loss += loss.item()

    # Print training progress
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")


Epoch [1/10], Loss: 397.7444
Epoch [2/10], Loss: 289.6809
Epoch [3/10], Loss: 237.1895
Epoch [4/10], Loss: 200.8008
Epoch [5/10], Loss: 175.2042
Epoch [6/10], Loss: 156.3070
Epoch [7/10], Loss: 142.3131
Epoch [8/10], Loss: 131.0663
Epoch [9/10], Loss: 121.9321
Epoch [10/10], Loss: 113.5633


In [13]:
# STEP 13: Save Trained Model
torch.save(model.state_dict(), "risnet_colab.pth")
print("Model saved as risnet_colab.pth")


Model saved as risnet_colab.pth


In [26]:
# STEP 14: Test on Sample Image
# Define the correct test image path
test_image_path = "/content/dataset/ACOD-12K/Test/Imgs/bean_left_00045.png"


# Read the image
test_image = cv2.imread(test_image_path)


# Resize the image
test_image = cv2.resize(test_image, (256, 256))

# Convert to tensor
test_tensor = torch.tensor(test_image).permute(2, 0, 1).unsqueeze(0).float() / 255.0

# Move model to evaluation mode
model.eval()

# Run Model on Test Image
with torch.no_grad():
    prediction = model(test_tensor.to(device))

# Convert Prediction to Image
output = prediction.cpu().squeeze().numpy()
output = (output > 0.5).astype(np.uint8) * 255  # Convert to binary mask

# Save Output Mask
output_mask_path = "output_mask.jpg"
cv2.imwrite(output_mask_path, output)
print(f"Output mask saved as {output_mask_path}")



Output mask saved as output_mask.jpg
